## File exploration

In [2]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/gldas_noah/data/level_2/GLDAS_NOAH025_3H.A20201230.0000.021.nc4.SUB.nc4"

data = xr.open_dataset(file, engine="netcdf4")
data

<xarray.Dataset> Size: 7MB
Dimensions:               (time: 1, bnds: 2, lon: 1440, lat: 600)
Coordinates:
  * time                  (time) datetime64[ns] 8B 2020-12-30
  * lon                   (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.6 179.9
  * lat                   (lat) float32 2kB -59.88 -59.62 -59.38 ... 89.62 89.88
Dimensions without coordinates: bnds
Data variables:
    time_bnds             (time, bnds) datetime64[ns] 16B ...
    SoilTMP10_40cm_inst   (time, lat, lon) float32 3MB ...
    SoilTMP40_100cm_inst  (time, lat, lon) float32 3MB ...
Attributes: (12/20)
    CDI:                    Climate Data Interface version 1.9.8 (https://mpi...
    Conventions:            CF-1.6
    history:                created on date: 2021-04-23T11:12:57.241
    source:                 Noah_v3.6 forced with GDAS-AGRMET-GPCPv13rA1_202101
    institution:            NASA GSFC
    missing_value:          -9999.0
    ...                     ...
    SOUTH_WEST_CORNER_LAT:  -59.875
    SOUTH_WEST_CORNER_LON:  -179.875
    DX:                     0.25
    DY:                     0.25
    history_L34RS:          'Created by L34RS v1.4.4 @ NASA GES DISC on Janua...
    CDO:                    Climate Data Operators version 1.9.8 (https://mpi...

In [3]:
import xarray as xr

# Your dataset
data = xr.open_dataset(file)

# Method 1: Select by lat/lon coordinates (easiest)
lat_target = 52.5  # Berlin latitude
lon_target = 13.4  # Berlin longitude

# Get values for both layers at that location
temp_10_40 = data['SoilTMP10_40cm_inst'].sel(lat=lat_target, lon=lon_target, method='nearest')
temp_40_100 = data['SoilTMP40_100cm_inst'].sel(lat=lat_target, lon=lon_target, method='nearest')

print(f"Location: {lat_target}°N, {lon_target}°E")
print(f"10-40cm layer: {temp_10_40.values[0]:.2f} K")
print(f"40-100cm layer: {temp_40_100.values[0]:.2f} K")

Location: 52.5°N, 13.4°E
10-40cm layer: 275.09 K
40-100cm layer: 276.80 K


In [5]:
for i, time_val in enumerate(data.time):
    temp_data = data.isel(time=i).sel(lat=52.5, lon=13.4, method='nearest')
    print(f"Hour {i:2d} ({time_val.values}): "
          f"SoilTMP10_40cm_inst={temp_data['SoilTMP10_40cm_inst'].values:.2f}K, "
          f"SoilTMP40_100cm_inst={temp_data['SoilTMP40_100cm_inst'].values:.2f}K, ")

Hour  0 (2020-12-30T00:00:00.000000000): SoilTMP10_40cm_inst=275.09K, SoilTMP40_100cm_inst=276.80K, 


## Checking combine files imported from cluster

In [1]:
import os
import glob
from pathlib import Path
import pandas as pd

ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/combine")

REQUIRED_COLS = {
    "date","time","ts_station_k","ts_gldas",
    "lat","lon","elev","cc","lc",
    "land_cover_group","climate_group","temp_class","elevation_class",
    "lat_gldas","lon_gldas",
}

records = []
problems = []

for net_dir in sorted(ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name

    for st_dir in sorted(net_dir.iterdir()):
        if not st_dir.is_dir():
            continue
        station = st_dir.name

        for f in glob.glob(str(st_dir / "*.csv")):
            rec = {"network": network, "station": station, "file": f}
            try:
                df = pd.read_csv(f)
            except Exception as e:
                rec["status"] = "unreadable"
                rec["issue"] = f"read_error: {e}"
                records.append(rec)
                problems.append(rec)
                continue

            rec["status"] = "ok"
            missing = [c for c in REQUIRED_COLS if c not in df.columns]
            if missing:
                rec["status"] = "missing_columns"
                rec["issue"] = "missing: " + ",".join(missing)
                problems.append(rec)

            if df.empty:
                rec["status"] = "empty_file"
                rec["issue"] = "no rows"
                problems.append(rec)
            else:
                n_valid_station = df["ts_station_k"].notna().sum() if "ts_station_k" in df else 0
                n_valid_gldas  = df["ts_gldas"].notna().sum()   if "ts_gldas"  in df else 0
                if n_valid_station == 0 and n_valid_gldas == 0:
                    rec["status"] = "no_valid_ts"
                    rec["issue"] = "ts_station_k & ts_gldas all NaN"
                    problems.append(rec)

            records.append(rec)

df_all = pd.DataFrame(records)

n_networks = df_all["network"].nunique()
n_stations = df_all[["network","station"]].drop_duplicates().shape[0]
n_files    = len(df_all)

print(f"Networks: {n_networks}")
print(f"Stations: {n_stations}")
print(f"Files:    {n_files}")

print("\nProblematic files by status:")
print(df_all["status"].value_counts())

# write reports
df_all.to_csv(ROOT / "audit_gldas_combine_listing.csv", index=False)
if problems:
    pd.DataFrame(problems).to_csv(ROOT / "audit_gldas_combine_problems.csv", index=False)
    print("\nSaved detailed problems to audit_gldas_combine_problems.csv")
else:
    print("\nAll files have required columns and at least one valid ts_station_k or ts_gldas.")

Networks: 34
Stations: 1173
Files:    1173

Problematic files by status:
status
ok    1173
Name: count, dtype: int64

All files have required columns and at least one valid ts_station_k or ts_gldas.


## Computing SD and correlation for gldas_noah sub surface layers combine files

In [2]:
import pandas as pd
from pathlib import Path

# Root dirs
GLDAS_COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/combine")
OUT_SUMMARY        = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/gldas_noah_insitu_sd_corr_summary.csv")

results = []

for network_dir in sorted(GLDAS_COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    network = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        station = station_dir.name

        csv_files = list(station_dir.glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV in {station_dir}, skipping.")
            continue

        csv_path = csv_files[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[ERROR] Cannot read {csv_path}: {e}")
            results.append({
                "network": network,
                "station": station,
                "n_points": 0,
                "sd_insitu": None,
                "sd_gldas": None,
                "sd_ratio_gldas_insitu": None,
                "corr_gldas_insitu": None,
                "pass_sd_filter": False,
                "pass_corr_filter": False,
            })
            continue

        # Optional: build datetime column
        if "date" in df.columns and "time" in df.columns and "datetime" not in df.columns:
            df["datetime"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce"
            )

        # Required columns
        if not {"ts_station_k", "ts_gldas"}.issubset(df.columns):
            print(f"[WARN] Missing ts_station_k or ts_gldas in {csv_path}, skipping station.")
            continue

        # Pairwise valid values only
        sub = df[["ts_station_k", "ts_gldas"]].dropna()
        n_points = len(sub)

        if n_points < 2:
            print(f"[WARN] Not enough valid points for {network}/{station} ({n_points}), skipping stats.")
            sd_insitu = sd_gldas = ratio = corr = None
            pass_sd = pass_corr = False
        else:
            sd_insitu = sub["ts_station_k"].std()
            sd_gldas  = sub["ts_gldas"].std()
            ratio = sd_gldas / sd_insitu if sd_insitu and sd_insitu > 0 else None
            corr  = sub["ts_station_k"].corr(sub["ts_gldas"])

            # Filters (same as MERRA2)
            pass_sd   = (ratio is not None) and (0.1 <= ratio <= 3)
            pass_corr = (corr is not None) and (corr >= 0.5)

        results.append({
            "network": network,
            "station": station,
            "n_points": n_points,
            "sd_insitu": sd_insitu,
            "sd_gldas": sd_gldas,
            "sd_ratio_gldas_insitu": ratio,
            "corr_gldas_insitu": corr,
            "pass_sd_filter": pass_sd,
            "pass_corr_filter": pass_corr,
        })

# Build DataFrame
summary_df = pd.DataFrame(results)

# Round numeric columns
num_cols = ["sd_insitu", "sd_gldas", "sd_ratio_gldas_insitu", "corr_gldas_insitu"]
for col in num_cols:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].round(3)

summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved summary to {OUT_SUMMARY}")


[WARN] Not enough valid points for SCAN/Combate (0), skipping stats.
[WARN] Not enough valid points for SCAN/MooseInc (0), skipping stats.
[WARN] Not enough valid points for SCAN/SchorGarden (0), skipping stats.
[WARN] Not enough valid points for SCAN/SunleafNursery (0), skipping stats.
[WARN] Not enough valid points for SCAN/UpperBethlehem (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/BeaverPass (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/Bourne (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/BrownTop (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/CayusePass (1), skipping stats.
[WARN] Not enough valid points for SNOTEL/ClearLake (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/ColumbiaBasin (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/CorduroyFlat (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/DefianceMines (0), skipping stats.
[WARN] Not enough valid points for SNOTEL/G

In [3]:
import pandas as pd
from pathlib import Path

summary_path = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/gldas_noah_insitu_sd_corr_summary.csv")

df = pd.read_csv(summary_path)

# Total stations
total = len(df)

# Stations that pass both SD and correlation filters
pass_both = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]

# Stations failing at least one filter
fail_any = df[(df["pass_sd_filter"] == False) | (df["pass_corr_filter"] == False)]

print(f"Total stations          : {total}")
print(f"Pass both (SD & corr)   : {len(pass_both)}")
print(f"Fail SD or corr (or both): {len(fail_any)}")

# Optional: counts per network
print("\nPer network (pass both):")
print(pass_both.groupby("network")["station"].count())

print("\nPer network (fail any):")
print(fail_any.groupby("network")["station"].count())

# NEW: print failing station names per network
print("\nFailing stations by network:")
for network, sub in fail_any.groupby("network"):
    station_list = sorted(sub["station"].unique())
    print(f"{network}: {station_list} = {len(station_list)}")


Total stations          : 1173
Pass both (SD & corr)   : 1105
Fail SD or corr (or both): 68

Per network (pass both):
network
ARM                   14
BIEBRZA_S-1           18
BNZ-LTER               8
Berlin                 1
COSMOS-UK             49
CTP_SMTMN             57
CW3E                   8
DAHRA                  1
FLUXNET-AMERIFLUX      5
FMI                   10
FR_Aqui                5
HOAL                  29
HOBE                  30
LABFLUX                1
MAQU                  16
MOL-RAO                2
NAQU                  10
NGARI                 20
ORACLE                 1
RISMA                 23
ROMPS                  5
Ru_CFR                 2
SCAN                 200
SKKU                   1
SMN-SDR               33
SMOSMANIA             22
SNOTEL               395
STEMS                  2
TERENO                 5
TWENTE                25
USCRN                 88
WEGENERNET            11
XMS-CAT                8
Name: station, dtype: int64

Per network (fail an

## test: checking the standard deviation and correlation between the station and product

- using combine file (gldas_in_situ) exported from cluster

In [1]:
import pandas as pd

# Path you gave
csv_path = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/combine/Anthony/ARM_Anthony_insitu_gldas_soil_temperature.csv"

# Read CSV
df = pd.read_csv(csv_path)

# Build a datetime index just for clarity (optional)
df["datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str))

# Keep only rows where both GLDAS and in-situ are valid
sub = df[["datetime", "ts_station_k", "ts_gldas"]].dropna(subset=["ts_station_k", "ts_gldas"])

if len(sub) < 2:
    print("Not enough data points to compute SD/correlation.")
else:
    # Standard deviations
    sd_insitu = sub["ts_station_k"].std()
    sd_gldas = sub["ts_gldas"].std()

    # SD ratio: GLDAS vs in-situ
    ratio_gldas = sd_gldas / sd_insitu if sd_insitu > 0 else None

    # Pearson correlation between GLDAS and in-situ
    corr_gldas = sub["ts_station_k"].corr(sub["ts_gldas"])

    print(f"Station: ARM / Anthony")
    print(f"Number of paired GLDAS–in-situ points: {len(sub)}")
    print(f"SD(in-situ)        = {sd_insitu:.4f} K")
    print(f"SD(GLDAS)          = {sd_gldas:.4f} K")
    print(f"SD ratio (GLDAS/insitu) = {ratio_gldas:.4f}")
    print(f"Correlation         = {corr_gldas:.4f}")

    # Optional: apply the paper's thresholds just for GLDAS
    if ratio_gldas is not None:
        if ratio_gldas < 0.1 or ratio_gldas > 3:
            print("→ SD ratio outside [0.1, 3]: would be flagged by SD filter (for GLDAS).")
        else:
            print("→ SD ratio within [0.1, 3]: passes SD filter (for GLDAS).")

    if corr_gldas < 0.5:
        print("→ Correlation < 0.5: would be flagged by correlation filter (for GLDAS).")
    else:
        print("→ Correlation ≥ 0.5: passes correlation filter (for GLDAS).")


Station: ARM / Anthony
Number of paired GLDAS–in-situ points: 14043
SD(in-situ)        = 6.4358 K
SD(GLDAS)          = 7.4822 K
SD ratio (GLDAS/insitu) = 1.1626
Correlation         = 0.9696
→ SD ratio within [0.1, 3]: passes SD filter (for GLDAS).
→ Correlation ≥ 0.5: passes correlation filter (for GLDAS).


## Taking pixel mean

In [4]:
import os
import glob
import pandas as pd
from collections import Counter
from pathlib import Path


# ---------------------------------------------------------------------
# Paths for GLDAS NOAH sub-surface
# ---------------------------------------------------------------------

COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/combine"
OUT_ROOT     = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/pixel_mean"
SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/gldas_noah_insitu_sd_corr_summary.csv"


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def majority(series):
    """Most common non-NaN value in a series."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return Counter(vals).most_common(1)[0][0]


def classify_elevation(elev):
    """Classify elevation using DEM-I/II/III/IV thresholds."""
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"

    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"


def load_pass_stations(summary_path):
    """
    Read GLDAS summary CSV and return a set of (network, station) pairs
    that passed BOTH SD and correlation filters.
    """
    df = pd.read_csv(summary_path)
    ok = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]
    return set(zip(ok["network"], ok["station"]))


# ---------------------------------------------------------------------
# Standard processing for moderate-size networks
# ---------------------------------------------------------------------

def process_network(network, pass_stations):
    base_in    = os.path.join(COMBINE_ROOT, network)
    base_out   = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network: {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    all_records = []
    used_station_dirs = []

    # Load station data, but only for stations that passed filters
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            print(f"  Skipping {network}/{station_name} (failed filter in summary).")
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            # Required columns for GLDAS sub-surface
            required_cols = {
                "date", "time",
                "ts_station_k", "ts_gldas",
                "lat", "lon", "elev",
                "cc", "lc",
                "land_cover_group", "climate_group", "temp_class",
                "elevation_class",
                "lat_gldas", "lon_gldas",
            }
            if not required_cols.issubset(df.columns):
                print(f"  [WARN] Missing required columns in {f}, skipping this file.")
                continue

            df["station"] = station_name
            all_records.append(df)
            used_station_dirs.append(station_dir)

    if not all_records:
        print(f"No valid CSV files found under {base_in}/* (after filtering), skipping.")
        return

    df_all = pd.concat(all_records, ignore_index=True)
    print(f"Loaded {len(set(used_station_dirs))} stations after filtering, total rows: {len(df_all)}")

    # DENSE PIXELS: at least 2 stations in same (lat_gldas, lon_gldas)
    pixel_station_counts = (
        df_all.groupby(["lat_gldas", "lon_gldas"])["station"]
             .nunique()
             .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_gldas", "lon_gldas"]]
    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    n_sparse_files = 0
    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # DENSE
    print("Processing dense pixels (pixel-mean files)...")
    df_dense_all = df_all.merge(dense_pixels, on=["lat_gldas", "lon_gldas"], how="inner")
    print("Rows in dense pixels (before grouping):", len(df_dense_all))

    stations_in_dense = set()

    # Also read existing dense files to populate stations_in_dense (idempotent)
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    if not df_dense_all.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_gldas"]
            plon = pix["lon_gldas"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            # Skip this pixel if its dense file already exists
            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_gldas={plat}, lon_gldas={plon} (file exists).")
                continue

            df_pix = df_dense_all[
                (df_dense_all["lat_gldas"] == plat) &
                (df_dense_all["lon_gldas"] == plon)
            ].copy()

            if df_pix.empty:
                continue

            print(f"  Dense pixel at lat_gldas={plat}, lon_gldas={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref   = g_valid["ts_station_k"].mean()
                    ts_gldas = g_valid["ts_gldas"].iloc[0]

                    lat_val   = majority(g_valid["lat"])
                    lon_val   = majority(g_valid["lon"])
                    cc_val    = majority(g_valid["cc"])
                    lc_val    = majority(g_valid["lc"])
                    lcg_val   = majority(g_valid["land_cover_group"])
                    clim_val  = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref   = pd.NA
                    ts_gldas = pd.NA
                    lat_val = lon_val = cc_val = lc_val = lcg_val = clim_val = tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_gldas": ts_gldas,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_gldas"]    = pd.to_numeric(df_pix_mean["ts_gldas"],    errors="coerce").round(3)

            df_pix_mean["lat_gldas"] = plat
            df_pix_mean["lon_gldas"] = plon
            df_pix_mean["stations"]  = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")
    else:
        print("No dense pixels found to process.")

    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")

    # SPARSE: station-wise files, excluding stations used in any dense pixel
    print("Writing sparse (station-wise) files (excluding dense stations)...")
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue

        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Stations found (raw): {n_stations_found}")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


# ---------------------------------------------------------------------
# SNOTEL-specific processing (memory-safe) for GLDAS
# ---------------------------------------------------------------------

def process_network_snotel(pass_stations):
    network = "SNOTEL"
    base_in    = os.path.join(COMBINE_ROOT, network)
    base_out   = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network (chunked): {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    # 1) FIRST PASS: determine dense pixels using minimal columns
    pixel_station_pairs = []

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f, usecols=["lat_gldas", "lon_gldas"])
            except Exception as e:
                print(f"  [WARN] Could not read minimal cols from {f}: {e}")
                continue

            pix_unique = df.dropna(subset=["lat_gldas", "lon_gldas"]).drop_duplicates()
            for _, row in pix_unique.iterrows():
                pixel_station_pairs.append(
                    (row["lat_gldas"], row["lon_gldas"], station_name)
                )

    if not pixel_station_pairs:
        print("No pixel/station info found for SNOTEL, skipping.")
        return

    df_pix_st = pd.DataFrame(
        pixel_station_pairs,
        columns=["lat_gldas", "lon_gldas", "station"]
    )

    pixel_station_counts = (
        df_pix_st.groupby(["lat_gldas", "lon_gldas"])["station"]
        .nunique()
        .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_gldas", "lon_gldas"]]

    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    stations_in_dense = set()

    # Existing dense files
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # 2) SECOND PASS: process each dense pixel separately
    if not dense_pixels.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_gldas"]
            plon = pix["lon_gldas"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_gldas={plat}, lon_gldas={plon} (file exists).")
                continue

            # stations that ever fall into this pixel
            stations_pix = df_pix_st[
                (df_pix_st["lat_gldas"] == plat) &
                (df_pix_st["lon_gldas"] == plon)
            ]["station"].unique()

            df_pix_all = []

            for station_name in stations_pix:
                if (network, station_name) not in pass_stations:
                    continue

                station_dir = os.path.join(base_in, station_name)
                csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
                if not csv_files:
                    continue

                for f in csv_files:
                    try:
                        df = pd.read_csv(f)
                    except Exception as e:
                        print(f"  [WARN] Could not read {f}: {e}")
                        continue

                    required_cols = {
                        "date", "time", "ts_station_k", "ts_gldas",
                        "lat_gldas", "lon_gldas",
                        "lat", "lon",
                        "cc", "lc",
                        "land_cover_group", "climate_group", "temp_class", "elev"
                    }
                    if not required_cols.issubset(df.columns):
                        continue

                    df = df[
                        (df["lat_gldas"] == plat) &
                        (df["lon_gldas"] == plon)
                    ].copy()

                    if df.empty:
                        continue

                    df["station"] = station_name
                    df_pix_all.append(df)

            if not df_pix_all:
                continue

            df_pix = pd.concat(df_pix_all, ignore_index=True)

            print(f"  Dense pixel at lat_gldas={plat}, lon_gldas={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref   = g_valid["ts_station_k"].mean()
                    ts_gldas = g_valid["ts_gldas"].iloc[0]

                    lat_val   = majority(g_valid["lat"])
                    lon_val   = majority(g_valid["lon"])
                    cc_val    = majority(g_valid["cc"])
                    lc_val    = majority(g_valid["lc"])
                    lcg_val   = majority(g_valid["land_cover_group"])
                    clim_val  = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref   = pd.NA
                    ts_gldas = pd.NA
                    lat_val = lon_val = cc_val = lc_val = lcg_val = clim_val = tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_gldas": ts_gldas,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_gldas"]    = pd.to_numeric(df_pix_mean["ts_gldas"],    errors="coerce").round(3)
            df_pix_mean["lat_gldas"] = plat
            df_pix_mean["lon_gldas"] = plon
            df_pix_mean["stations"]  = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")

    else:
        print("No dense pixels found for SNOTEL.")

    # 3) SPARSE: per-station outputs for stations not in dense pixels
    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")
    print("Writing sparse (station-wise) files (excluding dense stations)...")

    n_sparse_files = 0

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

if __name__ == "__main__":
    pass_stations = load_pass_stations(SUMMARY_PATH)

    network_dirs = sorted(
        d for d in glob.glob(os.path.join(COMBINE_ROOT, "*"))
        if os.path.isdir(d)
    )
    networks = [os.path.basename(d) for d in network_dirs]

    print("Networks found:", networks)
    for net in networks:
        if net == "SNOTEL":
            process_network_snotel(pass_stations)
        else:
            process_network(net, pass_stations)


Networks found: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'LABFLUX', 'MAQU', 'MOL-RAO', 'NAQU', 'NGARI', 'ORACLE', 'OZNET', 'RISMA', 'ROMPS', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'STEMS', 'TERENO', 'TWENTE', 'USCRN', 'WEGENERNET', 'XMS-CAT']

=== Network: ARM ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/combine/ARM
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/pixel_mean/ARM
Found 14 station directories.
Loaded 14 stations after filtering, total rows: 258713
Total unique pixels: 14
Dense pixels (n_stations >= 2): 0
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 0
No dense pixels found to process.
Stations that appear in dense pixels: 0
Writing sparse (station-wise) files (excluding dense stations)...
Finished sparse files. Processed 14 station 

## checking the pixel mean files

In [10]:
import os
import glob
import pandas as pd
from collections import defaultdict

# ---------------------------------------------------------------------
# Paths (GLDAS NOAH)
# ---------------------------------------------------------------------

SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/gldas_noah_insitu_sd_corr_summary.csv"
PIXEL_MEAN_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/pixel_mean"

# ---------------------------------------------------------------------
# 1. Load station list from summary
# ---------------------------------------------------------------------

summary_df = pd.read_csv(SUMMARY_PATH)

summary_ok = summary_df[
    (summary_df["pass_sd_filter"] == True) &
    (summary_df["pass_corr_filter"] == True)
]

summary_stations = set(zip(summary_ok["network"], summary_ok["station"]))

print(f"Total stations in summary (passed filters): {len(summary_stations)}")

# ---------------------------------------------------------------------
# 2. Collect stations appearing in dense and sparse
# ---------------------------------------------------------------------

stations_in_dense = set()
stations_in_sparse = set()

network_dirs = sorted(
    d for d in glob.glob(os.path.join(PIXEL_MEAN_ROOT, "*"))
    if os.path.isdir(d)
)
networks = [os.path.basename(d) for d in network_dirs]
print("Networks found in pixel_mean:", networks)

for network in networks:
    base_net  = os.path.join(PIXEL_MEAN_ROOT, network)
    dense_dir = os.path.join(base_net, "dense")
    sparse_dir = os.path.join(base_net, "sparse")

    # ---- Dense: station list from "stations" column ----
    if os.path.exists(dense_dir):
        dense_files = glob.glob(os.path.join(dense_dir, "*.csv"))
        for f in dense_files:
            try:
                df = pd.read_csv(f, usecols=["stations"])
            except Exception as e:
                print(f"[WARN] Could not read stations from dense file {f}: {e}")
                continue

            if df.empty or "stations" not in df.columns:
                continue

            stations_str = str(df["stations"].iloc[0])
            if not stations_str or stations_str.lower() == "nan":
                continue

            for st in stations_str.split(","):
                st = st.strip()
                if st:
                    stations_in_dense.add((network, st))

    # ---- Sparse: infer station from filename ----
    if os.path.exists(sparse_dir):
        sparse_files = glob.glob(os.path.join(sparse_dir, "*.csv"))
        for f in sparse_files:
            fname = os.path.basename(f)
            # Strip .csv
            name_no_ext = fname[:-4] if fname.endswith(".csv") else fname

            # Remove "<network>_" prefix if present
            prefix = network + "_"
            if name_no_ext.startswith(prefix):
                # Example:
                #   network = "CTP_SMTMN"
                #   name_no_ext = "CTP_SMTMN_L35_insitu_gldas_soil_temperature"
                #   tail = "L35_insitu_gldas_soil_temperature"
                tail = name_no_ext[len(prefix):]
            else:
                # If pattern is different, just use the full name_no_ext
                tail = name_no_ext

            # Station is the first token before the next underscore (before "_insitu")
            # e.g. "L35_insitu_gldas_soil_temperature" -> "L35"
            station_from_file = tail.split("_")[0]

            stations_in_sparse.add((network, station_from_file))

# ---------------------------------------------------------------------
# 3. Compute coverage and missing stations
# ---------------------------------------------------------------------

only_dense = summary_stations & stations_in_dense - stations_in_sparse
only_sparse = summary_stations & stations_in_sparse - stations_in_dense
both = summary_stations & stations_in_dense & stations_in_sparse
missing = summary_stations - stations_in_dense - stations_in_sparse

print("\n===== OVERALL STATION COVERAGE =====")
print(f"Stations only in dense:   {len(only_dense)}")
print(f"Stations only in sparse:  {len(only_sparse)}")
print(f"Stations in both:         {len(both)}")
print(f"Stations missing (neither dense nor sparse): {len(missing)}")

# Optional: per-network breakdown
per_network_counts = defaultdict(lambda: {"only_dense": 0, "only_sparse": 0, "both": 0, "missing": 0})

for net, st in only_dense:
    per_network_counts[net]["only_dense"] += 1
for net, st in only_sparse:
    per_network_counts[net]["only_sparse"] += 1
for net, st in both:
    per_network_counts[net]["both"] += 1
for net, st in missing:
    per_network_counts[net]["missing"] += 1

print("\n===== PER-NETWORK SUMMARY =====")
for net in sorted(per_network_counts.keys()):
    c = per_network_counts[net]
    print(
        f"{net}: "
        f"only_dense={c['only_dense']}, "
        f"only_sparse={c['only_sparse']}, "
        f"both={c['both']}, "
        f"missing={c['missing']}"
    )

# ---------------------------------------------------------------------
# 4. Print and save missing stations explicitly
# ---------------------------------------------------------------------

print("\n===== MISSING STATIONS (in summary but not in dense or sparse) =====")
print(f"Total missing: {len(missing)}\n")

for net, st in sorted(missing):
    print(f"{net},{st}")

if missing:
    missing_df = pd.DataFrame(sorted(missing), columns=["network", "station"])
    out_csv = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/gldas_noah/missing_stations_gldas.csv"
    missing_df.to_csv(out_csv, index=False)
    print(f"\nSaved missing stations to {out_csv}")


Total stations in summary (passed filters): 1105
Networks found in pixel_mean: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'LABFLUX', 'MAQU', 'MOL-RAO', 'NAQU', 'NGARI', 'ORACLE', 'OZNET', 'RISMA', 'ROMPS', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'STEMS', 'TERENO', 'TWENTE', 'USCRN', 'WEGENERNET', 'XMS-CAT']

===== OVERALL STATION COVERAGE =====
Stations only in dense:   501
Stations only in sparse:  604
Stations in both:         0
Stations missing (neither dense nor sparse): 0

===== PER-NETWORK SUMMARY =====
ARM: only_dense=0, only_sparse=14, both=0, missing=0
BIEBRZA_S-1: only_dense=18, only_sparse=0, both=0, missing=0
BNZ-LTER: only_dense=6, only_sparse=2, both=0, missing=0
Berlin: only_dense=0, only_sparse=1, both=0, missing=0
COSMOS-UK: only_dense=2, only_sparse=47, both=0, missing=0
CTP_SMTMN: only_dense=56, only_sparse=1, both=0, missing=0
CW3E: only_dense=6, o